# Class 4. Classical ML Refresher Through a Quantum Lens

EVA: Quantum Machine Learning | ZHAW | Pavel Sulimov

---

Goals of this practice session:

1. Use matrix-form notation for linear and logistic models.
2. Understand why XOR needs a feature map.
3. Compute and interpret Gram matrices.
4. Compare linear and nonlinear kernel baselines before moving to quantum feature maps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

---
## Part 1: Math tasks

### M4.1. Matrix-form prediction and sigmoid

Given
\[
X=\begin{bmatrix}
1 & 0.5\\
0.2 & 1.2\\
1.5 & 1.0
\end{bmatrix},
\quad
w=\begin{bmatrix}1.1\\-0.8\end{bmatrix},
\quad b=-0.1,
\]
compute $z=Xw+b\mathbf{1}$ and $\sigma(z)$.

In [ ]:
X = np.array([
    [1.0, 0.5],
    [0.2, 1.2],
    [1.5, 1.0],
])
w = np.array([1.1, -0.8])
b = -0.1

z = X @ w + b
sigma = 1.0 / (1.0 + np.exp(-z))

print("z:", np.round(z, 6))
print("sigma(z):", np.round(sigma, 6))

### M4.2. XOR lifting and separability check

Use
\(
\phi(x_1,x_2)=(x_1,x_2,x_1x_2)
\)
and verify that the hyperplane
\(
s=x_1+x_2-2x_3-0.5
\)
separates XOR classes.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

X_lift = np.column_stack([X_xor[:, 0], X_xor[:, 1], X_xor[:, 0] * X_xor[:, 1]])
s = X_lift[:, 0] + X_lift[:, 1] - 2 * X_lift[:, 2] - 0.5

print("Lifted points:\n", X_lift)
print("Signed values s:", s)
print("Class 0 signs:", s[y_xor == 0])
print("Class 1 signs:", s[y_xor == 1])

### M4.3. Polynomial kernel Gram matrix for XOR

Compute
\(
K(x,y)=(x^\top y + 1)^2
\)
for all XOR pairs.

In [ ]:
K = (X_xor @ X_xor.T + 1.0) ** 2
print(K)

---
## Part 2: Programming tasks

### P4.1. Linear vs RBF SVM on two-moons

In [ ]:
X, y = make_moons(n_samples=240, noise=0.20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svc_lin = SVC(kernel="linear", C=1.0)
svc_rbf = SVC(kernel="rbf", C=1.0, gamma=1.0)

svc_lin.fit(X_train_s, y_train)
svc_rbf.fit(X_train_s, y_train)

acc_lin = accuracy_score(y_test, svc_lin.predict(X_test_s))
acc_rbf = accuracy_score(y_test, svc_rbf.predict(X_test_s))

print(f"Linear SVM accuracy: {acc_lin:.3f}")
print(f"RBF SVM accuracy:    {acc_rbf:.3f}")

### P4.2. Visualize decision boundaries

In [ ]:
def plot_boundary(ax, model, X_data, y_data, title):
    x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
    y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 220), np.linspace(y_min, y_max, 220))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    zz = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
    ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap="coolwarm", s=20)
    ax.set_title(title)
    ax.grid(True, alpha=0.2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_boundary(axes[0], svc_lin, X_test_s, y_test, "Linear SVM")
plot_boundary(axes[1], svc_rbf, X_test_s, y_test, "RBF SVM")
plt.tight_layout()
plt.show()

### P4.3. Iris baseline: logistic regression vs kernel SVM

In [ ]:
iris = load_iris()
X_iris = iris.data[:, :2]
y_iris = (iris.target == 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

scaler2 = StandardScaler()
X_tr_s = scaler2.fit_transform(X_tr)
X_te_s = scaler2.transform(X_te)

clf_lr = LogisticRegression(max_iter=2000)
clf_rbf = SVC(kernel="rbf", C=1.0, gamma=1.0)

clf_lr.fit(X_tr_s, y_tr)
clf_rbf.fit(X_tr_s, y_tr)

acc_lr = accuracy_score(y_te, clf_lr.predict(X_te_s))
acc_rbf = accuracy_score(y_te, clf_rbf.predict(X_te_s))

print(f"Logistic regression accuracy: {acc_lr:.3f}")
print(f"RBF SVM accuracy:             {acc_rbf:.3f}")

---
## Summary

Linear models and kernels answer different geometric questions. Before adding a quantum feature map, we should always benchmark strong classical baselines (linear and nonlinear kernels) under the same preprocessing and data split.